In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Getting started with JAX and NumPyro

From [JAX Quickstart](https://jax.readthedocs.io/en/latest/notebooks/quickstart.html):
> JAX is NumPy on the CPU, GPU, and TPU, with great automatic differentiation for high-performance machine learning research.

From [Getting Started with NumPyro](https://num.pyro.ai/en/stable/getting_started.html):
> NumPyro is a lightweight probabilistic programming library that provides a NumPy backend for Pyro. We rely on JAX for automatic differentiation and JIT compilation to GPU / CPU. 

## Installation

If you load this repository in codespace everyting will already be installed.  If you're working on a local machine, in most cases a simple
```bash
pip install numpyro
```
will do the trick.  The behavior is unpredictable on Windows, however.  If you run into issues you can consider [this solution](https://github.com/cloudhan/jax-windows-builder) (thanks for finding this Jared Lawrence!), but a more robust solution may be to use [Windows Subsystem for Linux (WSL)](https://learn.microsoft.com/en-us/windows/wsl/install).  If you go that route, know that there is a VSCode extension to help it play nicely with VSCode [here](https://code.visualstudio.com/docs/remote/wsl) (thanks Ben Mannix!).

In [ ]:
from jax import random
import jax.numpy as jnp
import numpyro.distributions as dist

Let's start with drawing some samples from a Gaussian distribution.  First let's read the docs for the implementation of the normal distribution.

In [ ]:
dist.Normal?

...and for the `.sample()` method of the distribution object.

In [ ]:
rv = dist.Normal(0, 1)
rv.sample?

One change off the bat when working with `NumPyro` and `JAX` is the need to explicitly manage the random number generator (RNG).  Let's create an RNG key and generate a 1-D array of 10,000 samples from the distribution. 

In [ ]:
rng_key = random.PRNGKey(0)
samples = rv.sample(rng_key, (10000,))

In [ ]:
sns.displot(samples, kind='kde');

## Inference w/ NumPyro

The real power of `NumPyro` shines when we use it to conduct inference.  Let's generate some synthetic 1-D data from a normal distribution, then construct a forward model of a Gaussian distribution to describe and fit to that data.

In [ ]:
import numpyro
import arviz as az  # for plotting and diagnostics

In [ ]:
data = dist.Normal().sample(random.PRNGKey(0), (50,))
sns.displot(data)

Models are constructed as functions, whose arguments provide a means to specify data, specify assumptions, etc. Within the function we establish model parameters (the things we want to infer) by defining their **prior distributions** (what we "know" about them before collecting data, which is often "nothing").  We'll use a uniform prior for the mean of the Gaussian distribution (let's say from -10 to 10), and a half-normal distribution with a standard deviation of 10).  The last thing to define is the **likelihood**, which is where we establish the probabilistic description of the data and provide any observations.

In [ ]:
def model(xdata=None):
    # model parameters and their priors
    μ =  numpyro.sample("μ", dist.Uniform(low=-10, high=10))
    σ = numpyro.sample("σ", dist.HalfNormal(scale=10))

    # Likelihood: the probabilistic description of the data, and observations
    obs = numpyro.sample("obs", dist.Normal(μ, σ), obs=xdata)

In [ ]:
from numpyro import infer # where MCMC routines live

In [ ]:
# Generate a starting key, which we'll split for later operations.
rng_key = random.PRNGKey(0)
rng_key, rng_key_ = random.split(rng_key)

First we construct a "kernel", which describes _how_ an MCMC chain is updated.  We'll use the No-U-Turn sampler (NUTS), a Hamiltonian-MC-based sampler.

In [ ]:
kernel = infer.NUTS(model)

Now let's construct an MCMC object using that kernel, specifying that we'll use 1000 steps to tune the sampler followed by another 5000 steps that we'll use to estimate the posterior distribution.

In [ ]:
num_samples = 5000
mcmc = infer.MCMC(kernel, num_warmup=1000, num_samples=num_samples)

Now that we've constructed the MCMC object, it's time to run it.  Now is when we plug in any arguments for the model, in this case, just the data.

In [ ]:
mcmc.run(rng_key_, xdata=data)

We can use `.print_summary()` to print some summary statistics

In [ ]:
mcmc.print_summary()

### Reading that table

Two of those columns are diagnostics rather than results, and they are the ones to look at
**first** — before the parameter estimates mean anything.

**`n_eff` — the effective sample size.** We met this in week 3: consecutive MCMC samples are
correlated, so a chain of $N$ points carries the information of rather fewer independent
draws. `n_eff` is that smaller number. NumPyro asked for 5000 samples; if `n_eff` comes back
as a few hundred for some parameter, then that parameter's uncertainty is set by the few
hundred, not the five thousand.

**Use `n_eff`, not `num_samples`, when you quote a Monte Carlo error.** The standard error on
a posterior mean goes as $\sigma/\sqrt{n_\mathrm{eff}}$.

**`r_hat` — the Gelman–Rubin statistic.** NumPyro can run several chains from different
starting points. If they have all converged to the same distribution, the variance *between*
chains should match the variance *within* each one, and their ratio $\hat{R}$ should be
close to 1. Values noticeably above 1 mean the chains have not mixed — they are still
exploring different parts of the space and disagree about the answer.

**Rule of thumb: $\hat{R} < 1.01$ is usually taken as acceptable.** Larger values mean run
longer, or fix the model.

A low `n_eff` and a healthy `r_hat` are different complaints. `r_hat` says *the chains
disagree*; `n_eff` says *they agree, but inefficiently*. The first is a correctness problem.
The second is a cost problem — and it is the one that quietly widens your error bars if you
ignore it.


and extract the samples

In [ ]:
samples = mcmc.get_samples()
samples

Trace plots are a very useful diagnostic tool, giving us a fairly comprehensive picture of how the chain explored parameter space, and what the marginal (i.e., 1-D) posterior distributions for each parameter look like.

In [ ]:
az.plot_trace(mcmc);

Finally, let's choose some samples from the chain (at random) reconstruct the normal distribution they correspond to, and compare them to the data.

In [ ]:
sns.displot(data, stat='density')
x = jnp.linspace(-5, 5, 100)

for i in random.choice(rng_key, num_samples, (5,)):
    plt.plot(
        x,
        jnp.exp(dist.Normal(samples['μ'][i],
                            samples['σ'][i]).log_prob(x)),
        ls='--')

This gives us a sense of our uncertainty in the underlying distribution.  Let's plot a bunch more to make this clearer.

In [ ]:
sns.displot(data, stat='density')

for i in random.choice(rng_key, num_samples, (100,)):
    plt.plot(
        x,
        jnp.exp(dist.Normal(samples['μ'][i],
                            samples['σ'][i]).log_prob(x)),
        color='k',
        alpha=0.05)